In [1]:
import heapq

# Trạng thái đích của 15-puzzle (ô trống đại diện bằng số 0 ở cuối)
# (1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0)
GOAL_STATE = tuple(list(range(1, 16)) + [0])

class PuzzleNode:
    def __init__(self, state, parent=None, action=None, g=0, h=0):
        self.state = state      # Tuple 1D chứa 16 phần tử
        self.parent = parent    # Node cha để truy vết đường đi
        self.action = action    # Hành động dẫn đến node này (Lên, Xuống, Trái, Phải)
        self.g = g              # Chi phí từ trạng thái ban đầu (số bước)
        self.h = h              # Chi phí ước lượng đến đích (Manhattan)
        self.f = g + h          # Tổng chi phí f(n)

    def __lt__(self, other):
        # Nạp chồng toán tử < để heapq sắp xếp theo f(n)
        return self.f < other.f

def get_manhattan_distance(state):
    """Tính tổng khoảng cách Manhattan cho tất cả các ô số (trừ ô 0)"""
    distance = 0
    for i in range(16):
        val = state[i]
        if val != 0:
            # Tọa độ đích của ô val (0-indexed)
            target_x = (val - 1) // 4
            target_y = (val - 1) % 4
            # Tọa độ hiện tại
            curr_x = i // 4
            curr_y = i % 4
            # Cộng dồn khoảng cách
            distance += abs(target_x - curr_x) + abs(target_y - curr_y)
    return distance

def get_successors(node):
    """Sinh các trạng thái kế tiếp từ việc di chuyển ô trống (0)"""
    successors = []
    state = list(node.state)
    zero_idx = state.index(0)

    # Tọa độ 2D của ô trống
    curr_x = zero_idx // 4
    curr_y = zero_idx % 4

    # Các hướng di chuyển của ô trống
    moves = {
        'Lên': (curr_x - 1, curr_y),
        'Xuống': (curr_x + 1, curr_y),
        'Trái': (curr_x, curr_y - 1),
        'Phải': (curr_x, curr_y + 1)
    }

    for action, (new_x, new_y) in moves.items():
        # Kiểm tra xem nước đi có hợp lệ (nằm trong bảng 4x4) không
        if 0 <= new_x < 4 and 0 <= new_y < 4:
            new_idx = new_x * 4 + new_y
            new_state = state[:]

            # Hoán đổi ô trống với ô kề cạnh
            new_state[zero_idx], new_state[new_idx] = new_state[new_idx], new_state[zero_idx]
            new_state_tuple = tuple(new_state)

            # Tính toán chi phí
            g = node.g + 1
            h = get_manhattan_distance(new_state_tuple)

            successors.append(PuzzleNode(new_state_tuple, node, action, g, h))

    return successors

def a_star_search(start_state):
    """Thuật toán A* giải 15-puzzle"""
    h_start = get_manhattan_distance(start_state)
    start_node = PuzzleNode(start_state, parent=None, action=None, g=0, h=h_start)

    open_list = []
    heapq.heappush(open_list, start_node)

    # Tập hợp (set) lưu các trạng thái đã duyệt để tránh lặp (dùng tuple vì tuple có thể hash)
    closed_set = set()

    nodes_explored = 0

    while open_list:
        # Lấy node có f(n) nhỏ nhất
        current_node = heapq.heappop(open_list)
        nodes_explored += 1

        # Kiểm tra trạng thái đích
        if current_node.state == GOAL_STATE:
            print(f"Đã duyệt {nodes_explored} trạng thái.")

            # Truy vết đường đi
            path = []
            while current_node.parent:
                path.append((current_node.action, current_node.state))
                current_node = current_node.parent
            path.reverse()
            return path

        if current_node.state in closed_set:
            continue

        closed_set.add(current_node.state)

        # Mở rộng các trạng thái con
        for child in get_successors(current_node):
            if child.state not in closed_set:
                heapq.heappush(open_list, child)

    return None # Không tìm thấy đường đi

def print_board(state):
    """Hàm hỗ trợ in bảng 4x4 ra màn hình"""
    for i in range(0, 16, 4):
        row = state[i:i+4]
        print(" ".join(f"{str(x).rjust(2, ' ')}" if x != 0 else "  " for x in row))
    print("-" * 15)

# --- CHẠY THỬ NGHIỆM ---
if __name__ == "__main__":
    # Chọn một trạng thái bắt đầu có thể giải được trong vài bước để test nhanh
    # Trạng thái này cách đích 3 bước (Di chuyển ô 0: Trái, Xuống, Xuống)
    START_STATE = (
        1,  2,  3,  4,
        5,  6,  0,  8,
        9, 10,  7, 11,
       13, 14, 15, 12
    )

    print("Trạng thái ban đầu:")
    print_board(START_STATE)

    print("Đang giải bằng thuật toán A*...")
    solution_path = a_star_search(START_STATE)

    if solution_path:
        print(f"Tìm thấy giải pháp sau {len(solution_path)} bước!")
        for step, (action, state) in enumerate(solution_path, 1):
            print(f"Bước {step}: Di chuyển ô trống sang {action}")
            print_board(state)
    else:
        print("Không tìm thấy giải pháp!")

Trạng thái ban đầu:
 1  2  3  4
 5  6     8
 9 10  7 11
13 14 15 12
---------------
Đang giải bằng thuật toán A*...
Đã duyệt 4 trạng thái.
Tìm thấy giải pháp sau 3 bước!
Bước 1: Di chuyển ô trống sang Xuống
 1  2  3  4
 5  6  7  8
 9 10    11
13 14 15 12
---------------
Bước 2: Di chuyển ô trống sang Phải
 1  2  3  4
 5  6  7  8
 9 10 11   
13 14 15 12
---------------
Bước 3: Di chuyển ô trống sang Xuống
 1  2  3  4
 5  6  7  8
 9 10 11 12
13 14 15   
---------------


In [3]:
import heapq

def a_star_graph_search(graph, heuristics, start, goal):
    """
    Thuật giải A* tìm đường đi ngắn nhất trên đồ thị.

    :param graph: Dictionary biểu diễn đồ thị (danh sách kề và trọng số).
    :param heuristics: Dictionary chứa giá trị h(n) ước lượng từ mỗi đỉnh đến đích.
    :param start: Đỉnh bắt đầu.
    :param goal: Đỉnh đích.
    :return: Danh sách các đỉnh tạo thành đường đi ngắn nhất, hoặc None nếu không có đường.
    """
    # Hàng đợi ưu tiên chứa các Tuple: (f_score, node)
    open_list = []
    heapq.heappush(open_list, (heuristics[start], start))

    # Bảng lưu chi phí thực tế g(n) từ start đến một đỉnh n
    g_scores = {node: float('inf') for node in graph}
    g_scores[start] = 0

    # Bảng lưu vết để tái tạo đường đi
    came_from = {}

    while open_list:
        # Lấy đỉnh có f(n) nhỏ nhất ra khỏi hàng đợi
        current_f, current_node = heapq.heappop(open_list)

        # Nếu đã đạt đến đỉnh đích, tiến hành truy vết đường đi
        if current_node == goal:
            path = []
            while current_node in came_from:
                path.append(current_node)
                current_node = came_from[current_node]
            path.append(start)
            return path[::-1]  # Đảo ngược mảng để có thứ tự từ Start -> Goal

        # Duyệt qua các đỉnh kề (neighbor) của đỉnh hiện tại
        for neighbor, weight in graph[current_node].items():
            # Tính g(n) tạm thời nếu đi qua current_node
            tentative_g_score = g_scores[current_node] + weight

            # Nếu g(n) mới này tốt hơn (nhỏ hơn) g(n) cũ đã biết của neighbor
            if tentative_g_score < g_scores[neighbor]:
                # Cập nhật đường đi và chi phí
                came_from[neighbor] = current_node
                g_scores[neighbor] = tentative_g_score

                # Tính f(n) = g(n) + h(n) và đẩy vào hàng đợi
                f_score = tentative_g_score + heuristics[neighbor]
                heapq.heappush(open_list, (f_score, neighbor))

    return None  # Trả về None nếu duyệt hết mà không tới được đích

# --- CHẠY THỬ NGHIỆM ---
if __name__ == "__main__":
    # 1. Định nghĩa đồ thị bằng danh sách kề (đỉnh: {đỉnh_kề: chi_phí_g})
    graph = {
        'A': {'B': 2, 'E': 3},
        'B': {'A': 2, 'C': 1, 'G': 9},
        'C': {'B': 1, 'D': 2},
        'D': {'C': 2, 'G': 1},
        'E': {'A': 3, 'D': 4},
        'G': {'B': 9, 'D': 1}
    }

    # 2. Định nghĩa hàm Heuristic h(n) (Chi phí ước lượng từ đỉnh n đến đỉnh đích 'G')
    # Lưu ý: Trong A* chuẩn, Heuristic phải "admissible" (không bao giờ đánh giá quá cao chi phí thực tế)
    heuristics = {
        'A': 6,
        'B': 4,
        'C': 2,
        'D': 1,
        'E': 4,
        'G': 0  # Heuristic tại đích luôn bằng 0
    }

    start_node = 'A'
    goal_node = 'G'

    print(f"Đang tìm đường đi từ {start_node} đến {goal_node}...")
    path = a_star_graph_search(graph, heuristics, start_node, goal_node)

    if path:
        print("Đường đi ngắn nhất tìm được:", " -> ".join(path))

        # Tính tổng chi phí của đường đi này
        total_cost = 0
        for i in range(len(path) - 1):
            total_cost += graph[path[i]][path[i+1]]
        print(f"Tổng chi phí: {total_cost}")
    else:
        print("Không tồn tại đường đi giữa hai đỉnh này!")

Đang tìm đường đi từ A đến G...
Đường đi ngắn nhất tìm được: A -> B -> C -> D -> G
Tổng chi phí: 6
